# 25 Export NPZ

目标：按固定日期、WFICN 与 59 特征顺序构造张量，并逐位置回读验证。

- 输入：24 号 model-ready 长面板
- 正式输出：kaniel_1980_2024_model_ready.npz 及索引/元数据
- 主键：dates×wficn×feature_names
- 上游依赖：24
- smoke 输出与正式输出隔离；报告会明确标注运行模式。

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import json
import os
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from src.atomic_write import write_csv_atomic, write_json_atomic, write_npz_atomic
from src.config_utils import config_hash
from src.release_utils import file_sha256
from src.runtime_paths import output_path, upstream_path
from src.schema_v3_alpha_audit import compare_npz_members

features=CONFIG["feature_directions"]["feature_order"]
macro_features=list(PARAMS["macro_feature_names"])
assert len(features)==59 and len(set(features))==59
assert macro_features==["Sentiment","CFNAI"]
assert not {"rret","alpha","Sentiment","CFNAI"} & set(features)
assert PARAMS["label_name"]=="alpha"
panel=pd.read_parquet(upstream_path(ROOT,RUN_MODE,"24_build_model_ready_panel","outputs/panels/model_ready_panel.parquet"))
assert "alpha" in panel.columns and "rret" not in panel.columns
for column in ["target_month","feature_month","sentiment_source_month","cfnai_source_month"]:
    panel[column]=pd.to_datetime(panel[column]).dt.to_period("M").dt.to_timestamp()
dates=np.array(sorted(panel["target_month"].dropna().unique()),dtype="datetime64[M]")
def wficn_sort(value):
    try: return (0,int(float(value)))
    except Exception: return (1,str(value))
wficn=np.array(sorted(panel["wficn"].dropna().astype(str).unique(),key=wficn_sort),dtype=str)
date_pos={pd.Timestamp(value):index for index,value in enumerate(dates.astype("datetime64[ns]"))}
fund_pos={value:index for index,value in enumerate(wficn)}
X=np.full((len(dates),len(wficn),len(features)),np.nan,dtype=np.float32)
y=np.full((len(dates),len(wficn)),np.nan,dtype=np.float32)
X_macro=np.full((len(dates),len(macro_features)),np.nan,dtype=np.float32)
date_index=panel["target_month"].map(date_pos).to_numpy(dtype=np.int64)
fund_index=panel["wficn"].astype(str).map(fund_pos).to_numpy(dtype=np.int64)
panel_values=panel[features].apply(pd.to_numeric,errors="coerce").to_numpy(dtype=np.float32)
panel_y=pd.to_numeric(panel["alpha"],errors="coerce").to_numpy(dtype=np.float32)
X[date_index,fund_index,:]=panel_values
y[date_index,fund_index]=panel_y

macro_monthly=panel[[
    "target_month","feature_month","sentiment_source_month","cfnai_source_month",
    "Sentiment","CFNAI",
]].drop_duplicates()
assert not macro_monthly.duplicated(["target_month"]).any()
macro_date_index=macro_monthly["target_month"].map(date_pos).to_numpy(dtype=np.int64)
macro_values=macro_monthly[macro_features].apply(pd.to_numeric,errors="coerce").to_numpy(dtype=np.float32)
X_macro[macro_date_index,:]=macro_values

missing=np.float32(PARAMS["missing_export_value"])
X_export=np.where(np.isnan(X),missing,X).astype(np.float32)
X_macro_export=np.where(np.isnan(X_macro),missing,X_macro).astype(np.float32)
y_export=np.where(np.isnan(y),missing,y).astype(np.float32)
feature_months=dates-1
assert X.shape==(len(dates),len(wficn),59)
assert X_macro.shape==(len(dates),2)
assert y.shape==(len(dates),len(wficn))
if RUN_MODE=="full":
    assert len(dates)==540 and len(wficn)==6207
    assert str(dates.min())=="1980-01" and str(dates.max())=="2024-12"
assert np.array_equal(feature_months,dates-1)

out=output_path(ROOT,RUN_MODE,"25_export_npz","outputs/npz/kaniel_1980_2024_model_ready.npz")
meta=output_path(ROOT,RUN_MODE,"25_export_npz","outputs/npz/kaniel_1980_2024_metadata.json")
fidx=output_path(ROOT,RUN_MODE,"25_export_npz","outputs/npz/feature_names.csv")
midx=output_path(ROOT,RUN_MODE,"25_export_npz","outputs/npz/macro_feature_names.csv")
widx=output_path(ROOT,RUN_MODE,"25_export_npz","outputs/npz/wficn_index.csv")
didx=output_path(ROOT,RUN_MODE,"25_export_npz","outputs/npz/date_index.csv")
write_npz_atomic(
    out,
    X=X_export,
    X_macro=X_macro_export,
    y=y_export,
    dates=dates,
    feature_months=feature_months,
    wficn=wficn,
    feature_names=np.array(features),
    macro_feature_names=np.array(macro_features),
)
execution_ctx=execution_context()
if execution_ctx=="production" and RUN_MODE=="full":
    archive_value=os.getenv("KANIEL_ARCHIVE_ROOT","").strip()
    assert archive_value,"KANIEL_ARCHIVE_ROOT is required for schema v3 cross-version validation"
    archive_root=(ROOT/archive_value).resolve()
    old_npz_path=archive_root/"outputs/npz/kaniel_1980_2024_model_ready.npz"
    old_metadata_path=archive_root/"outputs/npz/kaniel_1980_2024_metadata.json"
    assert old_npz_path.is_file() and old_metadata_path.is_file()
    npz_equivalence=compare_npz_members(old_npz_path,out)
    cross_version_exact={
        row["key"]:bool(row["exact"]) for row in npz_equivalence["arrays"]
    }
    assert npz_equivalence["key_order_exact"],npz_equivalence
    invariant_keys=[
        "X_macro","y","dates","feature_months","wficn",
        "feature_names","macro_feature_names",
    ]
    assert all(cross_version_exact[key] for key in invariant_keys),npz_equivalence
    with np.load(old_npz_path) as old_archive,np.load(out) as new_archive:
        assert np.array_equal(
            old_archive["X"]==np.float32(PARAMS["missing_export_value"]),
            new_archive["X"]==np.float32(PARAMS["missing_export_value"]),
        )
elif execution_ctx=="clean_rebuild_compute" and RUN_MODE=="full":
    archive_value=None
    archive_root=None
    old_npz_path=None
    old_metadata_path=None
    npz_equivalence=None
    cross_version_exact=None
elif RUN_MODE=="smoke":
    archive_value=None
    archive_root=None
    old_npz_path=None
    old_metadata_path=None
    npz_equivalence=None
    cross_version_exact=None
else:
    raise RuntimeError("Notebook 25 cannot run in clean_rebuild_compare context.")

metadata={
  "schema_version":PARAMS["schema_version"],
  "data_release_version":PARAMS["data_release_version"],
  "implementation_version":PARAMS["implementation_version"],
  "release_name":PARAMS["release_name"],
  "production_dataset_definition":"Kaniel core 59 features with project alpha label and Sentiment/CFNAI macro extension",
  "run_mode":RUN_MODE,
  "label_name":"alpha",
  "label_semantic_name":"rolling_carhart_abnormal_return",
  "label_formula_version":"rolling_36m_ffc4_beta_without_intercept_subtraction",
  "legacy_project_label_name":"rret",
  "label_rename_only":True,
  "label_values_changed":False,
  "official_model_label_name":"alpha",
  "official_raw_return_field":"rret",
  "official_raw_return_alias":"official_rret_raw",
  "official_same_semantic_reference":"alpha",
  "official_same_name_collision_resolved":True,
  "core_feature_count":59,
  "macro_feature_count":2,
  "total_predictor_count":61,
  "sample_target_month_start":str(dates.min()),
  "sample_target_month_end":str(dates.max()),
  "n_target_months":int(len(dates)),
  "fund_count":int(len(wficn)),
  "feature_count":59,
  "feature_names":list(features),
  "feature_names_order":list(features),
  "macro_feature_names":macro_features,
  "macro_source_lag_months":{"Sentiment":1,"CFNAI":2},
  "macro_alignment":{"Sentiment":"observation_month=t-1","CFNAI":"observation_month=t-2"},
  "macro_vintage_policy":{
    "Sentiment":"current_published_historical_series_no_realtime_vintage",
    "CFNAI":"current_vintage_with_two_month_target_lag",
  },
  "macro_release_limitation":{
    "Sentiment":"historical realtime availability not observed",
    "CFNAI":"two-month lag avoids direct release leakage but later revisions remain",
  },
  "macro_missing_target_months":{
    "Sentiment":[],
    "CFNAI":["1980-01","1980-02"],
  },
  "X_shape":list(X.shape),
  "X_macro_shape":list(X_macro.shape),
  "y_shape":list(y.shape),
  "dates_meaning":"target return month t",
  "feature_months_meaning":"formation month t-1 for the 59 core features",
  "wficn_order":wficn.tolist(),
  "missing_export_value":float(missing),
  "generated_utc":datetime.now(timezone.utc).isoformat(),
  "configuration_hash":config_hash(ROOT/"config"),
  "direction_config_version":int(CONFIG["feature_directions"]["version"]),
  "direction_status":CONFIG["feature_directions"]["status"],
  "pca_imputation":{
    "solver":"deterministic_direct_tsvd",
    "relative_singular_value_threshold":0.05,
    "n_components":10,
    "minimum_observed_features":10,
    "singular_value_rule":"retain s_i where s_i > rcond * s_max",
    "component_sign_rule":"largest_absolute_loading_positive",
    "observed_value_policy":"preserve_exactly",
    "missing_mask_policy":"preserve_exactly",
    "clipping":False,
    "rounding":False,
    "winsorization":False,
    "selection_basis":"five-salt pseudo-missing recovery, perturbation stability, effective rank, extreme values, and external ranking diagnostics",
    "official_parameter":False,
  },
  "SUV":{
    "method":"monthly_oos_suv_v1",
    "estimation_window":60,
    "window_semantics":"last_valid_stock_observations",
    "min_estimation_obs":40,
    "min_evaluation_obs":10,
    "gap_trading_days":0,
    "monthly_aggregation":"mean",
    "sigma_floor":1.0e-12,
    "exact_cpz_replication":False,
    "implementation_family":"Garfinkel-style out-of-sample abnormal volume",
    "legacy_method":"legacy_same_sample_zero",
    "legacy_method_deprecated":True,
  },
  "CF":{
    "status":"unresolved_wcapch_coverage",
    "raw_formula_modified":False,
    "raw_missing_rule_modified":False,
    "final_values_may_change_via_joint_pca":True,
  },
  "deterministic_numeric":{
    "holdings_sum":"stable_order_compensated_float64",
    "risk_regression":"deterministic_float64_qr_or_lstsq",
    "residual_sum_of_squares":"compensated_float64",
    "stock_feature_aggregation":"stable_order_compensated_float64",
    "critical_linear_algebra_threads":1,
  },
  "migration_from":{
    "schema_version":"3.0.0",
    "data_release_version":"3.1.0",
    "release":"kaniel_alpha_59core_2macro_v3.1",
    "reason":"replace mechanically zero same-sample SUV with monthly out-of-sample SUV",
  },
  "stage2_suv_manifest_sha256":os.getenv("KANIEL_STAGE2_SUV_MANIFEST_SHA256"),
  "v31_baseline_canonical_sha256":os.getenv("KANIEL_V31_BASELINE_CANONICAL_SHA256"),
  "v32_pca_input_canonical_sha256":os.getenv("KANIEL_V32_PCA_INPUT_CANONICAL_SHA256"),
  "candidate_A_id":os.getenv("KANIEL_CANDIDATE_A_ID"),
  "candidate_B_id":os.getenv("KANIEL_CANDIDATE_B_ID"),
  "candidate_A_vs_B_exact":False,
  "SUV_direction_v31":int(CONFIG["feature_directions"]["stock_directions"]["SUV"]),
  "SUV_direction_v32":None,
  "forbidden_read_count":None,
  "direction_match_status":"pending",
  "time_leakage_records":None,
  "formal_npz_usable":False,
  "formal_usability_state":(
      "clean_rebuild_candidate_not_published"
      if execution_ctx=="clean_rebuild_compute"
      else (
          "smoke_artifact_not_formal"
          if RUN_MODE=="smoke"
          else "pending_notebook_26_full_quality_audit"
      )
  ),
  "schema_v3_npz_sha256":file_sha256(out),
  "schema_v3_metadata_sha256":None,
  "schema_v3_metadata_hash_scope":"populated by Notebook 26 as canonical JSON excluding schema_v3_metadata_sha256",
}
if execution_ctx=="production" and RUN_MODE=="full":
    metadata.update({
      "migration_baseline_npz_sha256":"611dc08b16a568f1a44dab69746843bbeda04d2482ec81ba1ba6590a20c6d9e2",
      "migration_baseline_metadata_sha256":"4345a95e4a0ae039dc753b5748d354bebc728daef8cc47be0eaad07621571d43",
      "cross_version_y_exact_match_rate":1.0 if cross_version_exact["y"] else 0.0,
      "cross_version_X_exact_match_rate":1.0 if cross_version_exact["X"] else 0.0,
      "cross_version_X_macro_exact_match_rate":1.0 if cross_version_exact["X_macro"] else 0.0,
    })
elif execution_ctx=="clean_rebuild_compute" and RUN_MODE=="full":
    metadata.update({
      "execution_context":"clean_rebuild_compute",
      "candidate_npz_usable":False,
      "candidate_internal_audit_passed":False,
      "candidate_not_published":True,
      "comparison_pending":True,
    })
write_json_atomic(meta,metadata)
write_csv_atomic(pd.DataFrame({"feature_index":range(59),"feature_name":features}),fidx)
write_csv_atomic(pd.DataFrame({"macro_feature_index":range(2),"macro_feature_name":macro_features}),midx)
write_csv_atomic(pd.DataFrame({"wficn_index":range(len(wficn)),"wficn":wficn}),widx)
write_csv_atomic(pd.DataFrame({
    "date_index":range(len(dates)),
    "target_month":dates.astype(str),
    "feature_month":feature_months.astype(str),
}),didx)

with np.load(out) as archive:
    assert archive.files==[
        "X","X_macro","y","dates","feature_months","wficn","feature_names","macro_feature_names"
    ]
    assert archive["X"].shape==X_export.shape
    assert archive["X_macro"].shape==X_macro_export.shape
    assert archive["y"].shape==y_export.shape
    assert list(archive["macro_feature_names"].astype(str))==macro_features
    assert "alpha" not in archive.files
expected_X=np.where(np.isnan(panel_values),missing,panel_values)
expected_y=np.where(np.isnan(panel_y),missing,panel_y)
expected_macro=np.where(np.isnan(macro_values),missing,macro_values)
assert np.array_equal(X_export[date_index,fund_index,:],expected_X,equal_nan=True)
assert np.array_equal(y_export[date_index,fund_index],expected_y,equal_nan=True)
assert np.array_equal(X_macro_export[macro_date_index,:],expected_macro,equal_nan=True)

INPUTS_USED=["model_ready_panel.parquet"]
OUTPUTS_WRITTEN=[str(path.relative_to(ROOT)) for path in [out,meta,fidx,midx,widx,didx]]
INPUT_COUNTS={"panel_rows":len(panel)}
OUTPUT_COUNTS={"T":len(dates),"N":len(wficn),"F":59,"M":2}
PRIMARY_KEY=["date_index","wficn_index"]; DUPLICATE_KEY_COUNT=0
EXCLUDED_ROW_COUNT=0; EXCEPTION_FILE=None; STAGE_STATUS="passed"
STAGE_NOTES=[
    "-99.99 is introduced only during NPZ export.",
    "X_macro is monthly and has no WFICN axis.",
    (
        "Candidate remains unpublished and pending isolated Notebook 26 audit."
        if execution_ctx=="clean_rebuild_compute"
        else "Formal usability remains false until Notebook 26 validates and publishes the staged release."
    ),
]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "25_export_npz", stage="25", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
